# 05 — Modeling and Evaluation

This notebook builds baseline machine learning pipelines for customer churn prediction.

Main goals:
- prepare train and test sets
- build reproducible preprocessing pipelines
- compare multiple classification models
- evaluate model performance with suitable classification metrics

In [12]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path
import json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)



## 1. Load the dataset and configuration

The raw dataset is loaded again and the saved preprocessing decisions are reused to keep the modeling stage consistent with the previous notebook.

In [6]:
DATA_PATH = Path("../data/raw/telco.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (7043, 50)


,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [7]:
CONFIG_PATH = Path("../reports/feature_config.json")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    feature_config = json.load(f)

feature_config

{'target_column': 'target_churn',
 'dropped_columns': ['Customer Status',
  'Churn Category',
  'Churn Reason',
  'Churn Score',
  'Customer ID',
  'Country',
  'State',
  'City',
  'Zip Code',
  'Latitude',
  'Longitude',
  'Quarter',
  'CLTV',
  'Total Revenue',
  'Churn Label'],
 'numeric_features': ['Age',
  'Number of Dependents',
  'Population',
  'Number of Referrals',
  'Tenure in Months',
  'Avg Monthly Long Distance Charges',
  'Avg Monthly GB Download',
  'Monthly Charge',
  'Total Charges',
  'Total Refunds',
  'Total Extra Data Charges',
  'Total Long Distance Charges',
  'Satisfaction Score'],
 'categorical_features': ['Gender',
  'Under 30',
  'Senior Citizen',
  'Married',
  'Dependents',
  'Referred a Friend',
  'Offer',
  'Phone Service',
  'Multiple Lines',
  'Internet Service',
  'Internet Type',
  'Online Security',
  'Online Backup',
  'Device Protection Plan',
  'Premium Tech Support',
  'Streaming TV',
  'Streaming Movies',
  'Streaming Music',
  'Unlimited Data

## 2. Recreate the baseline modeling dataset

The target variable and the preprocessing decisions from the previous notebook are applied again here.

The saved configuration helps keep column exclusions and feature groups consistent, while the raw dataset is prepared again to keep this notebook reproducible and self-contained.

In [8]:
df["target_churn"] = df["Churn Label"].map({"No": 0, "Yes": 1})

object_cols = df.select_dtypes(include="object").columns.tolist()
df[object_cols] = df[object_cols].replace("None", np.nan)
df[object_cols] = df[object_cols].replace(r"^\s*$", np.nan, regex=True)

df[["Churn Label", "target_churn"]].head()

,Churn Label,target_churn
0,Yes,1
1,Yes,1
2,Yes,1
3,Yes,1
4,Yes,1


In [9]:
drop_cols = feature_config["dropped_columns"]

baseline_df = df.drop(columns=drop_cols).copy()

print("Baseline shape:", baseline_df.shape)
baseline_df.head()

Baseline shape: (7043, 36)


,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Population,Referred a Friend,Number of Referrals,...,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Satisfaction Score,target_churn
0,Male,78,No,Yes,No,No,0,68701,No,0,...,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,3,1
1,Female,74,No,Yes,Yes,Yes,1,55668,Yes,1,...,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,3,1
2,Male,71,No,Yes,No,Yes,3,47534,No,0,...,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,2,1
3,Female,78,No,Yes,Yes,Yes,1,27778,Yes,1,...,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2,1
4,Female,80,No,Yes,Yes,Yes,1,26265,Yes,1,...,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,2,1


## 2.1 Define features and target

The baseline dataset is split into input features (`X`) and target labels (`y`) for modeling.

In [10]:
X = baseline_df.drop(columns=["target_churn"])
y = baseline_df["target_churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 35)
y shape: (7043,)


## 2.2 Identify feature groups

Numeric and categorical feature groups are loaded from the saved preprocessing configuration.

These groups will be used to apply different preprocessing steps inside the modeling pipeline.

In [11]:
numeric_features = feature_config["numeric_features"]
categorical_features = feature_config["categorical_features"]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

Numeric features: 13
Categorical features: 22

Numeric columns:
['Age', 'Number of Dependents', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Satisfaction Score']

Categorical columns:
['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']


## 3. Train-test split

The dataset is divided into training and test sets.

The test set is reserved for final evaluation, while model comparison is supported with cross-validation on the training set.

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 35)
X_test : (1409, 35)
y_train: (5634,)
y_test : (1409,)


### Why cross-validation was used instead of a separate validation split

A separate validation set could have been created, but this project uses cross-validation on the training data instead.

Because the dataset is moderate in size, this approach allows more efficient use of the available training observations while still providing a more stable comparison across models.

The test set is kept untouched and reserved only for final evaluation.

## 4. Cross-validation setup

To make the model comparison more reliable, cross-validation is applied on the training set.

This allows the baseline models to be compared more robustly before the final test-set evaluation.

In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

## 5. Build preprocessing pipelines

Separate preprocessing steps are defined for numeric and categorical features.

These are combined using a `ColumnTransformer` to create a clean and reproducible workflow.

In [15]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [16]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore'))])

In [17]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Number of Dependents', 'Population',
                                  'Number of Referrals', 'Tenure in Months',
                                  'Avg Monthly Long Distance Charges',
                                  'Avg Monthly GB Download', 'Monthly Charge',
                                  'Total Charges', 'Total Refunds',
                                  'Total Extra Data Charges',
                                  'Total Long Distanc...
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['Gender', 'Under 30', 'Senior Citizen',
                                  'Married', 'Dependents', 'Referred a Friend',
                                  'Offer', 'Phone Service', 'Multiple Lines',
                                  'Internet Service', 'Internet Type',
                                  'Online Security', 'Online Backup',
                                  'Device Protection Plan',
                                  'Premium Tech Support', 'Streaming TV',
                                  'Streaming Movies', 'Streaming Music',
                                  'Unlimited Data', 'Contract',
                                  'Paperless Billing', 'Payment Method'])])

## Numeric Features
* **Median Imputation:** Chosen for robustness against outliers compared to mean imputation.
* **StandardScaler:** Applied to normalize feature magnitudes, ensuring a stable baseline for linear models.

## Categorical Features
* **Most Frequent (Mode) Imputation:** A simple baseline strategy for missing data.
* **One-Hot Encoding:** Used to convert categories to numeric vectors without introducing artificial ordinality.

## Decision Matrix

| Stage | Method | Rationale |
| :--- | :--- | :--- |
| **Num Impute** | Median | Robustness |
| **Num Scale** | StandardScaler | Feature parity |
| **Cat Impute** | Most Frequent | Simplicity |
| **Cat Encode** | One-Hot | No artificial order |
